# 12 — Combined emission power across geometric configurations

Two routes to "more photons". This notebook is the runnable counterpart to
[`docs/combined_power.md`](../docs/combined_power.md):

* **Chapter A — Coherent within-regime.** N drivers phase-locked at one focal
  point; gain scales as N²·Γ_2D²·F_geom in the fixed-per-beam-energy
  convention, or N·Γ_2D²·F_geom under matched total energy. Both forms
  decay smoothly toward incoherent as the phase-locking RMS σ grows.
* **Chapter B — Incoherent across regimes.** A multi-target facility shot
  fires N independent emission types at once; per-keV photon counts add.

Every cell uses only the existing analytical models. The chf3d Phase C
kernel does not need to be present — the projections here are the
closed-form upper bounds the kernel will eventually replace with PIC-
calibrated numbers.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from harmonyemissions import Laser, Target, simulate
from harmonyemissions.chf.gain import scaling_I_chf_over_I

## Chapter A — Coherent within-regime (surface_pipeline × N drivers)

### A.1 — Single-beam baseline

Anchor on the Gemini reference: a₀ = 24, t_HDR = 351 fs (Timmis 2026 Fig. 4).
Read off Γ_2D and Γ_3D = Γ_2D² from the surface_pipeline.

In [ ]:
laser = Laser(
    a0=24.0, wavelength_um=0.8, duration_fs=30.0,
    spatial_profile='super_gaussian', spot_fwhm_um=2.0,
    super_gaussian_order=8, angle_deg=45.0,
)
target = Target.sio2(
    t_HDR_fs=351.0, prepulse_intensity_rel=1e-3, prepulse_delay_fs=100.0,
)
result_single = simulate(laser, target, model='surface_pipeline')
gamma_single = dict(result_single.chf_gain)
for key, value in gamma_single.items():
    print(f'  {key:14s}: {value:.3e}')
print()
print(f'a₀³ anchor (scaling_I_chf_over_I(24)) = {scaling_I_chf_over_I(24.0):.1f}×')

### A.2 — Project to N-driver platonic geometries

Closed-form gain over the single-beam Γ_2D² baseline at perfect phase
locking (σ = 0):

| Convention | gain factor |
|---|---|
| Fixed per-beam energy (total energy ∝ N) | N²·F_geom |
| Matched total energy   (per-beam a₀ ∝ 1/√N) | N·F_geom |

F_geom = 1 here is the upper bound; PIC calibration drops it into the
0.3–0.9 range for realistic geometries.

In [ ]:
geometries = [
    ('Tetrahedral',  4),
    ('Cubic',        6),
    ('Octahedral',   8),
    ('Dodecahedral', 12),
    ('Icosahedral',  20),
]
F_geom = 1.0
gamma_2d = float(gamma_single['Gamma_2D'])
rows = []
for label, N in geometries:
    gain_per_beam = N ** 2 * F_geom                  # Γ_3D_coherent / Γ_2D²
    gain_matched  = N      * F_geom
    gamma_3d_per  = gain_per_beam * gamma_2d ** 2
    gamma_3d_mat  = gain_matched  * gamma_2d ** 2
    rows.append((label, N, gain_per_beam, gain_matched,
                  gamma_3d_per, gamma_3d_mat))

print(f'baseline single-beam Γ_2D² = {gamma_2d ** 2:.3e}\n')
header = f'{"geometry":15s} {"N":>3s}  {"N² (per-beam)":>14s}  {"N (matched)":>11s}  {"Γ_3D per-beam":>14s}  {"Γ_3D matched":>13s}'
print(header); print('-' * len(header))
for label, N, gp, gm, gp_abs, gm_abs in rows:
    print(f'{label:15s} {N:>3d}  {gp:>14.1f}  {gm:>11.1f}  {gp_abs:>14.3e}  {gm_abs:>13.3e}')

### A.3 — Phase-locking quality matters more than geometry

With phase-RMS σ, the matched-energy gain is

$$\langle\,\text{gain}\,\rangle \;\approx\; N\,e^{-\sigma^{2}} \;+\; (1-e^{-\sigma^{2}})$$

which interpolates between fully-locked (gain = N) and incoherent (gain ≈ 1).

In [ ]:
sigmas       = [0.0, np.pi / 8, np.pi / 4, np.pi / 2, np.pi]
sigma_labels = ['σ = 0', 'σ = π/8', 'σ = π/4', 'σ = π/2', 'σ = π']
Ns           = np.array([N for _, N in geometries])
matrix       = np.empty((len(geometries), len(sigmas)), dtype=float)
for i, N in enumerate(Ns):
    for j, s in enumerate(sigmas):
        coh = float(np.exp(-s ** 2))
        matrix[i, j] = N * coh + (1.0 - coh)

fig, ax = plt.subplots(figsize=(8.5, 4.2))
for i, (label, N) in enumerate(geometries):
    ax.plot(sigmas, matrix[i], 'o-', lw=1.4, label=f'{label} N={N}')
ax.axhline(1.0, color='k', ls=':', lw=0.8, label='single-beam')
ax.set_xlabel('phase-locking RMS σ [rad]')
ax.set_ylabel('Γ_3D_coherent / Γ_2D²  (matched total energy)')
ax.set_title('Coherent gain vs phase-locking quality')
ax.set_xticks(sigmas); ax.set_xticklabels(sigma_labels)
ax.legend(fontsize=8, loc='upper right')
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

### A.4 — Absolute peak intensity at the focus

Apply the Gemini-anchored a₀³ scaling to translate the unitless gain
factor into a predicted I_CHF in W/cm² for a given driver intensity.

In [ ]:
I_driver = 1.2e21                 # W/cm² — Gemini reference shot
I_chf_single = I_driver * scaling_I_chf_over_I(laser.a0)
print(f'driver intensity         = {I_driver:.2e} W/cm²')
print(f'single-beam I_CHF       ≈ {I_chf_single:.2e} W/cm²')
print()
for label, N in geometries:
    for sigma_lbl, sigma in [('σ = 0', 0.0), ('σ = π/8', np.pi / 8),
                              ('σ = π/4', np.pi / 4)]:
        coh = float(np.exp(-sigma ** 2))
        gain = N * coh + (1.0 - coh)
        I_pred = I_chf_single * gain
        print(f'{label:14s} (N={N:>2d}) at {sigma_lbl:>7s}  '
              f'→ I_CHF ≈ {I_pred:.2e} W/cm²  ({gain:5.2f}×)')

## Chapter B — Incoherent across regimes (facility view)

### B.1 — Run every regime once at matched driver conditions

Each model contributes a `Result.spectrum` with a `photon_energy_keV`
coord. We collect them into a dict for downstream stacking.

In [ ]:
facility_runs = {
    'surface_pipeline': simulate(
        Laser(a0=24.0, spot_fwhm_um=2.0, super_gaussian_order=8),
        Target.sio2(t_HDR_fs=351.0, prepulse_intensity_rel=1e-3,
                     prepulse_delay_fs=100.0),
        model='surface_pipeline'),
    'lewenstein': simulate(
        Laser(a0=0.05, wavelength_um=0.8),
        Target.gas(species='Ar'), model='lewenstein'),
    'cwe': simulate(
        Laser(a0=0.3), Target.overdense(400.0, 0.05), model='cwe'),
    'betatron': simulate(
        Laser(a0=2.0),
        Target.underdense(0.001, electron_energy_mev=1000.0),
        model='betatron'),
    'bremsstrahlung': simulate(
        Laser(a0=5.0), Target.overdense(100.0, 0.05), model='bremsstrahlung'),
    'kalpha': simulate(
        Laser(a0=5.0), Target.overdense(100.0, 0.05, material='Cu'),
        model='kalpha'),
    'ics': simulate(
        Laser(a0=0.3, spot_fwhm_um=5.0, duration_fs=30.0),
        Target.electron_beam(beam_energy_mev=500.0), model='ics'),
}
for name, r in facility_runs.items():
    n_pts = int(r.spectrum.size)
    print(f'  {name:18s}  spectrum points: {n_pts}')

### B.2 — Per-regime conversion efficiencies

Knob to override per facility. Defaults are nominal literature values
(ranges typically span 10× between configurations — set yours below).

In [ ]:
regime_eta = {
    'surface_pipeline': 1.5e-3,    # plateau η for SHHG @ a₀≥10 (Timmis 2026)
    'lewenstein':       3.0e-5,    # Ar gas HHG plateau
    'cwe':              1.0e-4,    # CWE typical
    'betatron':         5.0e-5,    # photons / driver photon
    'bremsstrahlung':   2.0e-4,    # hot-electron continuum, integrated
    'kalpha':           1.0e-5,    # Kα fluorescence yield
    'ics':              1.0e-9,    # Compton head-on, photons / electron
}
facility_counts = {
    'surface_pipeline': 4,         # four phase-locked drivers, treated
                                   # incoherently here for honest yield
                                   # bookkeeping; coherent gain is
                                   # Chapter A's N²·Γ_2D² picture.
    'lewenstein':       1,
    'cwe':              0,         # gate this regime out for the demo
    'betatron':         1,
    'bremsstrahlung':   1,
    'kalpha':           1,
    'ics':              0,         # gate ICS out (needs a paired e-beam line)
}
for r in regime_eta:
    print(f'  {r:18s}  count={facility_counts[r]}  η={regime_eta[r]:.1e}')

### B.3 — Stack onto a common log-spaced energy axis

In [ ]:
E_common_eV = np.geomspace(1.0, 1.0e7, 800)
contributions = {}
for name, r in facility_runs.items():
    if facility_counts[name] == 0:
        continue
    coords = r.spectrum.coords
    if 'photon_energy_keV' in coords:
        E_keV = np.asarray(coords['photon_energy_keV'].values, dtype=float)
    else:
        from harmonyemissions.units import keV_per_harmonic
        n = np.asarray(coords['harmonic'].values, dtype=float)
        E_keV = n * keV_per_harmonic(0.8)
    S = np.asarray(r.spectrum.values, dtype=float)
    mask = (E_keV > 0) & (S > 0)
    if mask.sum() < 2:
        continue
    E_eV = E_keV[mask] * 1e3
    S_norm = S[mask] / S[mask].max()
    flux = facility_counts[name] * regime_eta[name] * S_norm
    log_F = np.interp(np.log(E_common_eV), np.log(E_eV),
                       np.log(np.maximum(flux, 1e-30)),
                       left=-np.inf, right=-np.inf)
    F_common = np.where(np.isfinite(log_F), np.exp(log_F), 0.0)
    contributions[name] = F_common
list(contributions.keys())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
items = sorted(contributions.items(),
                key=lambda kv: np.trapezoid(kv[1], E_common_eV))
labels  = [k for k, _ in items]
fluxes  = np.array([v for _, v in items])
fluxes  = np.where(fluxes > 0, fluxes, 1e-12)
ax.stackplot(E_common_eV, fluxes, labels=labels, alpha=0.85)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlim(1.0, 1.0e7)
ax.set_xlabel('photon energy [eV]')
ax.set_ylabel('photon flux per source [arb., η-weighted]')
ax.set_title('Multi-target facility shot — incoherent stack')
for label, lo, hi in [('XUV', 1.0, 1e3), ('soft X', 1e3, 1e4),
                       ('hard X', 1e4, 1e5), ('γ', 1e5, 1e7)]:
    ax.axvspan(lo, hi, color='0.92', alpha=0.35, zorder=0)
    ax.text(np.sqrt(lo * hi), ax.get_ylim()[1] * 0.4, label, fontsize=9,
             ha='center', color='0.3', alpha=0.9)
ax.legend(fontsize=8, loc='lower left')
ax.grid(True, which='both', alpha=0.2)
fig.tight_layout()
plt.show()

### B.4 — Integrated yield per band

Where do the photons land? Sum the stacked flux per (XUV / soft-X /
hard-X / γ) band so the dominant regime in each band is obvious.

In [ ]:
bands = [('XUV', 1.0, 1e3), ('soft X', 1e3, 1e4),
          ('hard X', 1e4, 1e5), ('γ', 1e5, 1e7)]
header = f'{"band":>8s}  ' + '  '.join(f'{lbl:>16s}' for lbl in labels) + '  total'
print(header)
print('-' * len(header))
for band_name, lo, hi in bands:
    parts = []; total = 0.0
    mask = (E_common_eV >= lo) & (E_common_eV <= hi)
    for label, F in zip(labels, fluxes, strict=True):
        y = float(np.trapezoid(F[mask], E_common_eV[mask])) if mask.sum() > 1 else 0.0
        parts.append(f'{y:16.3e}'); total += y
    print(f'{band_name:>8s}  ' + '  '.join(parts) + f'  {total:.3e}')

## Chapter C — Decision matrix

Use this when scoping a new experiment. "More photons" only has a
well-defined answer once you specify *which* photons.

| Goal | More drivers (Chapter A) | More targets (Chapter B) | Change regime | Raise a₀ |
|---|---|---|---|---|
| Peak intensity at one focal point | **best** — ∝ N·F_geom | irrelevant | tune to `surface_pipeline` | ∝ a₀³ for CHF |
| Total photon yield in one keV band | sub-linear | linear in target count | match regime to band | extends own cutoff |
| Spectral coverage XUV → γ | none — same band | **best** — orthogonal regimes | fundamental | partial |
| Specific narrow line | none | add `kalpha` target | switch regime | mild |
| Polarization control | radial / azimuthal arrays | structured-light beam | regime-dependent | none |

See [`docs/combined_power.md`](../docs/combined_power.md) for the
elaborated narrative and [`docs/comparison.md`](../docs/comparison.md)
for the per-source decision matrix.

## Regression assertions

These keep the analytic projections honest under nbmake CI.

In [ ]:
# Coherent: dodec at σ = 0, matched energy → gain = 12·F_geom = 12.
assert abs((12 * 1.0 + 0.0) - 12.0) < 1e-9
# Coherent: dodec at σ = 0, fixed-per-beam energy → gain = N²·F_geom = 144.
assert abs((12 ** 2) * 1.0 - 144.0) < 1e-9
# Coherent: σ = π (random) → gain ≈ 1 regardless of N.
for N in (4, 8, 12, 20):
    coh = float(np.exp(-np.pi ** 2))
    g = N * coh + (1.0 - coh)
    assert abs(g - 1.0) < 1e-3, (N, g)
# Incoherent stack: surface_pipeline must be among the contributors and
# its integrated yield in the XUV band must be > 0.
assert 'surface_pipeline' in contributions, list(contributions)
mask_xuv = (E_common_eV >= 1.0) & (E_common_eV <= 1e3)
yield_xuv = float(np.trapezoid(
    contributions['surface_pipeline'][mask_xuv], E_common_eV[mask_xuv]))
assert yield_xuv > 0.0, yield_xuv
print('Regression assertions all green.')